# FLUX Phase 2.5: Ontological Bootstrapping & Analogical Mapping
## Complete Pipeline — Seed, Test, Demo, Upload

> **Goal:** Upgrade the Resonance Field to a sparse dynamic architecture,
> seed it with the structural map of human knowledge (ConceptNet),
> teach it the physics of logic (SNLI + GSM8K),
> and demonstrate analogical reasoning across unseen domains.

### What this notebook does:
1. **Setup** — Clone/pull repo, install deps, detect hardware
2. **Load Stack** — Phase 1 CSE + Phase 1.5 CWC + Phase 2 field weights
3. **Build Sparse Field** — Dynamic 256³ address space, inherits Phase 2 topology
4. **ConceptNet Seeding** — Ontological bootstrapping with growth monitoring
5. **SNLI Curriculum** — Contradiction/entailment geometry
6. **GSM8K Curriculum** — Step-by-step logical chain embedding
7. **Analogical Mapper** — Build and validate reasoning engine
8. **Test 1** — Sparse field integrity
9. **Test 2** — Contradiction repulsion
10. **Test 3** — Unseen logic routing
11. **Demo 1** — Sparse galaxy (VRAM efficiency)
12. **Demo 2** — Analogical leaps
13. **Results** — Generate RESULTS_PHASE_2_5.md
14. **Final** — Upload logs + checkpoint → HuggingFace & GitHub

### Dynamic Growth Config:
The field starts at 64³ (inherited from Phase 2) and grows as attractors fill it.
A checkpoint is saved at each tier transition. Configure below.

### Secrets Required:
- **`HF_TOKEN`** — HuggingFace write token

---
## Cell 1: Clone / Pull Repository

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Unseengap/FLUX.git"
WORK_DIR = Path("/kaggle/working/FLUX")

if WORK_DIR.exists() and (WORK_DIR / '.git').exists():
    print("ℹ Repo already exists — pulling latest changes...")
    result = subprocess.run(
        ['git', 'pull', '--ff-only'],
        cwd=str(WORK_DIR), capture_output=True, text=True
    )
    print(result.stdout or result.stderr)
    print("✓ Pulled latest")
else:
    print(f"ℹ Cloning {REPO_URL}...")
    subprocess.run(['git', 'clone', REPO_URL, str(WORK_DIR)], check=True)
    print("✓ Cloned successfully")

os.chdir(str(WORK_DIR))
print(f"\nWorking directory: {os.getcwd()}")
print(f"Files: {sorted(os.listdir('.'))}")

---
## Cell 2: Install Dependencies & Setup

In [ ]:
!pip install -q datasets rich faiss-cpu huggingface_hub matplotlib
!python setup.py

---
## Cell 3: Hardware, VRAM Audit, Secrets & Dynamic Growth Config

**Edit `MAX_TIER` and `GROWTH_TRIGGER` to control field growth behaviour.**

| MAX_TIER | Max dimensions | Dense equivalent | Recommended for |
|----------|---------------|------------------|-----------------|
| 0        | 64³            | ~0.5 GB          | Smoke test only |
| 1        | 96³            | ~1.8 GB          | 8 GB GPU        |
| 2        | 128³           | ~4.3 GB          | 12 GB GPU       |
| 3        | 160³           | ~8.4 GB          | 16 GB GPU       |
| 4        | 192³           | ~17 GB           | 24 GB GPU       |
| 5        | 256³           | ~34 GB           | 40 GB GPU       |

In [ ]:
import sys, torch, shutil, os
from pathlib import Path

# Clear Kaggle working directory to ensure fresh repo sync
WORK_DIR = Path("/kaggle/working/FLUX")
if WORK_DIR.exists():
    print("Clearing /kaggle/working/FLUX for fresh sync...")
    shutil.rmtree(WORK_DIR)
    print("✓ Cleared old repo directory")

# (Re)clone repo to working directory
REPO_URL = "https://github.com/Unseengap/FLUX.git"
os.system(f"git clone {REPO_URL} {WORK_DIR}")

sys.path.insert(0, str(WORK_DIR))
sys.path.insert(0, str(WORK_DIR / 'phases' / 'phase1'))
sys.path.insert(0, str(WORK_DIR / 'phases' / 'phase1_5'))
sys.path.insert(0, str(WORK_DIR / 'phases' / 'phase2'))
sys.path.insert(0, str(WORK_DIR / 'phases' / 'phase2_5'))

# ══════════════════════════════════════════
# DYNAMIC GROWTH CONFIG — EDIT THESE
# ══════════════════════════════════════════
MAX_TIER       = 5      # 0-5, see table above. 5 = attempt 256³
GROWTH_TRIGGER = 0.60   # Grow when field is 60% full
MAX_CONCEPTS   = 150000   # ConceptNet triples to seed (increased for >50% capacity)
MAX_SNLI       = 5000    # SNLI examples (increased)
MAX_GSM8K      = 2000    # GSM8K problems (increased)
# ══════════════════════════════════════════

from flux_utils import (
    get_device, get_hardware_info, PhaseLogger, _resolve_hf_token,
    load_checkpoint, save_checkpoint,
    upload_checkpoint_to_hf, upload_logs_to_hf, git_commit_and_push,
)
from growth_manager import GROWTH_TIERS

log    = PhaseLogger(phase=2.5)
log.separator("Phase 2.5: Ontological Bootstrapping & Analogical Mapping")
log.cell_start("Cell 3 — Hardware & Config")

device = get_device()
hw     = get_hardware_info()
for k, v in hw.items():
    log.info(f"{k}: {v}")
    print(f"  {k}: {v}")

# VRAM audit
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    vram_free_mb  = free / 1e6
    vram_total_mb = total / 1e6
    print(f"\n  VRAM: {vram_free_mb:.0f} MB free / {vram_total_mb:.0f} MB total")

    # Auto-cap MAX_TIER based on available VRAM (conservative)
    suggested_max = 0
    for tid, h, w, d, label in GROWTH_TIERS:
        # Sparse: assume 50K active locations at most
        sparse_needed = 50000 * 512 * 4 / 1e6
        if sparse_needed + 1500 < vram_free_mb:
            suggested_max = tid
    if suggested_max < MAX_TIER:
        print(f"  ⚠ MAX_TIER capped at {suggested_max} based on available VRAM")
        print(f"    (You set {MAX_TIER}, VRAM suggests {suggested_max})")
        print(f"    Keeping your setting — growth will stop if VRAM runs out.")
    else:
        print(f"  ✓ MAX_TIER={MAX_TIER} looks feasible given available VRAM")

# Print growth tier plan
print(f"\n  Growth plan (MAX_TIER={MAX_TIER}):")
for tid, h, w, d, label in GROWTH_TIERS:
    dense_gb   = h*w*d*512*4/1e9
    marker = "← target" if tid == MAX_TIER else ("← start" if tid == 0 else "")
    print(f"    Tier {tid}: {h}³  dense={dense_gb:.1f} GB  sparse~<1 GB  [{label}] {marker}")

token = _resolve_hf_token()
if token:
    log.success("HuggingFace token loaded")
    print("\n  ✓ HF token loaded")
else:
    log.warning("No HuggingFace token")
    print("\n  ⚠ No HF token — uploads will be skipped")

log.cell_end("Cell 3 — Hardware & Config", "PASS")

---
## Cell 4: Load Phase 1 CSE (frozen)

In [ ]:
log.cell_start("Cell 4 — Load Phase 1 CSE")

from cse import ContinuousSemanticEncoder

ckpt1 = load_checkpoint(1)
cse   = ContinuousSemanticEncoder(**ckpt1['config'])
cse.load_state_dict(ckpt1['state_dict'])
cse   = cse.to(device).eval()
for p in cse.parameters():
    p.requires_grad = False

n_params = sum(p.numel() for p in cse.parameters())
log.success(f"CSE loaded and frozen: {n_params:,} params")
print(f"  ✓ CSE loaded: {n_params:,} params (frozen)")

# Smoke test
with torch.no_grad():
    test_wave = cse.encode("the sparse field awakens")
assert test_wave.full.shape[-1] == 432
print(f"  ✓ CSE smoke test: wave shape {test_wave.full.shape}")

log.cell_end("Cell 4 — Load Phase 1 CSE", "PASS")

---
## Cell 5: Load Phase 1.5 CWC (frozen)

In [ ]:
log.cell_start("Cell 5 — Load Phase 1.5 CWC")

from causal_encoder import CausalWaveChainer
from causal_types import TOTAL_CAUSAL_DIM

ckpt15 = load_checkpoint(1.5)
cwc    = CausalWaveChainer(**ckpt15['config'], device=device).to(device)
cwc.load_state_dict(ckpt15['state_dict'])
cwc.eval()
for p in cwc.parameters():
    p.requires_grad = False

n_cwc = sum(p.numel() for p in cwc.parameters())
log.success(f"CWC loaded and frozen: {n_cwc:,} params")
print(f"  ✓ CWC loaded: {n_cwc:,} params (frozen)")

# Smoke test
with torch.no_grad():
    cw_test = cwc.forward(test_wave)
assert cw_test.full.shape[-1] == TOTAL_CAUSAL_DIM
print(f"  ✓ CWC smoke test: causal wave shape {cw_test.full.shape}")

log.cell_end("Cell 5 — Load Phase 1.5 CWC", "PASS")

---
## Cell 6: Build SparseResonanceField
Inherits SpatialProjection and wave_to_feature weights from Phase 2.
Migrates existing Phase 2 attractors to sparse registry.

In [ ]:
log.cell_start("Cell 6 — Build SparseResonanceField")

from dynamic_field import SparseResonanceField
from growth_manager import GROWTH_TIERS

# Starting tier — always 0 (64³, Phase 2 dimensions)
start_h, start_w, start_d = 64, 64, 64

field = SparseResonanceField(
    initial_h      = start_h,
    initial_w      = start_w,
    initial_d      = start_d,
    features       = 512,
    wave_dim       = 432,
    max_tier       = MAX_TIER,
    checkpoint_dir = str(WORK_DIR / 'checkpoints'),
    device         = device,
).to(device)

# Load Phase 2 weights
print("  Loading Phase 2 checkpoint...")
ckpt2 = load_checkpoint(2)
field.inherit_from_phase2(ckpt2['state_dict'])

status = field.active_capacity()
for k, v in status.items():
    if k != 'timestamp':
        log.info(f"{k}: {v}")

print(f"\n  Field initialized:")
print(f"    Dimensions:       {field.h}×{field.w}×{field.d}")
print(f"    Active locations: {field.registry.active_count():,}")
print(f"    Capacity:         {field.registry.capacity_fraction():.4%}")
print(f"    Sparse memory:    {field.registry.memory_mb():.1f} MB")
print(f"    Max tier:         {MAX_TIER} ({GROWTH_TIERS[MAX_TIER][1]}³)")

log.cell_end("Cell 6 — Build SparseResonanceField", "PASS")

---
## Cell 7: ConceptNet Seeding
Ontological bootstrapping — seeds the structural map of human knowledge
directly into the field. Growth checkpoints saved automatically.

In [ ]:
log.cell_start("Cell 7 — ConceptNet Seeding")

from concept_seeder import OntologicalSeeder, load_conceptnet_triples
from implication import ImplicationChainStore

impl_store = ImplicationChainStore(device=device)

# Load ConceptNet triples
print(f"  Loading up to {MAX_CONCEPTS} ConceptNet triples...")
triples = load_conceptnet_triples(path=None, max_triples=MAX_CONCEPTS)
print(f"  ✓ Loaded {len(triples)} triples")

seeder = OntologicalSeeder(cse, cwc, field, impl_store, device=device)

extra_state = {
    'cse_state':  cse.state_dict(),
    'cwc_state':  cwc.state_dict(),
}

seed_stats = seeder.seed_batch(
    triples,
    log_every    = max(10, len(triples) // 20),
    check_growth = True,
    extra_state  = extra_state,
)

log.metric("concepts_seeded",   seed_stats['total_seeded'])
log.metric("arrows_registered", seed_stats['arrows_registered'])
log.metric("antonyms_repelled", seed_stats['antonyms_repelled'])
log.metric("growth_events",     seed_stats['growth_events'])
log.metric("active_locations",  field.registry.active_count())
log.metric("current_tier",      field.growth_manager.current_tier)

print(f"\n  Field after seeding:")
print(f"    Dimensions:    {field.h}×{field.w}×{field.d}")
print(f"    Active locs:   {field.registry.active_count():,}")
print(f"    Capacity:      {field.registry.capacity_fraction():.4%}")
print(f"    Tier:          {field.growth_manager.current_tier}")
print(f"    Growth events: {seed_stats['growth_events']}")

log.cell_end("Cell 7 — ConceptNet Seeding", "PASS")

---
## Cell 8: SNLI Curriculum
Contradiction and entailment geometry embedded into field topology.

In [ ]:
log.cell_start("Cell 8 — SNLI Curriculum")

from reasoning_curriculum import CurriculumRunner

curriculum = CurriculumRunner(cse, cwc, field, impl_store, device=device)

snli_stats = curriculum.run_snli(
    examples     = None,  # Auto-download or fallback
    max_examples = MAX_SNLI,
    log_every    = max(20, MAX_SNLI // 10),
    check_growth = True,
)

log.metric("snli_entailment",    snli_stats['entailment'])
log.metric("snli_contradiction", snli_stats['contradiction'])
log.metric("snli_neutral",       snli_stats['neutral'])

print(f"\n  SNLI ingested: E={snli_stats['entailment']} C={snli_stats['contradiction']} N={snli_stats['neutral']}")

log.cell_end("Cell 8 — SNLI Curriculum", "PASS")

---
## Cell 9: GSM8K Curriculum
Step-by-step deductive reasoning geometry.

In [ ]:
log.cell_start("Cell 9 — GSM8K Curriculum")

gsm8k_stats = curriculum.run_gsm8k(
    problems     = None,
    max_problems = MAX_GSM8K,
    log_every    = max(10, MAX_GSM8K // 10),
    check_growth = True,
)

log.metric("gsm8k_problems",   gsm8k_stats['problems_ingested'])
log.metric("gsm8k_steps",      gsm8k_stats['steps_as_attractors'])
log.metric("gsm8k_arrows",     gsm8k_stats['arrows_created'])
log.metric("impl_store_total", len(impl_store.arrows))

print(f"\n  GSM8K ingested: {gsm8k_stats['problems_ingested']} problems, {gsm8k_stats['arrows_created']} arrows")
print(f"  ImplicationStore total arrows: {len(impl_store.arrows)}")

log.cell_end("Cell 9 — GSM8K Curriculum", "PASS")

---
## Cell 10: Save Checkpoint & Upload to HuggingFace

In [ ]:
log.cell_start("Cell 10 — Save & Upload Checkpoint")

checkpoint_state = {
    'format':   'FLUX_SPARSE',
    'version':  '0.1',
    'phase':    '2.5',
    'config': {
        'features':  512,
        'wave_dim':  432,
        'max_tier':  MAX_TIER,
    },
    'state_dict':      field.get_state_for_checkpoint(),
    'cse_state':       cse.state_dict(),
    'cwc_state':       cwc.state_dict(),
    'impl_store':      impl_store.save(),
    'seed_stats':      seed_stats,
    'snli_stats':      snli_stats,
    'gsm8k_stats':     gsm8k_stats,
    'growth_history':  [e.as_dict() for e in field.growth_manager.growth_events],
    'metrics': {
        'active_locations':  field.registry.active_count(),
        'capacity_fraction': field.registry.capacity_fraction(),
        'sparse_memory_mb':  field.registry.memory_mb(),
        'current_tier':      field.growth_manager.current_tier,
        'impl_arrows':       len(impl_store.arrows),
        'num_attractors':    field.num_attractors(),
    },
    'can_continue_learning': True,
}
save_checkpoint('2_5', checkpoint_state)

uploaded = upload_checkpoint_to_hf(phase='2_5', hf_token=token)
if uploaded:
    log.success("Checkpoint uploaded to HuggingFace Hub")
    print("  ✓ Checkpoint uploaded")
else:
    log.warning("Upload skipped")

upload_logs_to_hf(phase='2_5', hf_token=token)
log.cell_end("Cell 10 — Save & Upload Checkpoint", "PASS")

---
## Cell 11: Test 1 — Sparse Field Integrity

In [ ]:
log.cell_start("Cell 11 — Test 1: Sparse Field Integrity")
import os
os.chdir(str(WORK_DIR / 'phases' / 'phase2_5'))
!python test_phase2_5_test1.py
os.chdir(str(WORK_DIR))
log.cell_end("Cell 11 — Test 1", "PASS")

---
## Cell 12: Test 2 — Contradiction Repulsion

In [ ]:
log.cell_start("Cell 12 — Test 2: Contradiction Repulsion")
os.chdir(str(WORK_DIR / 'phases' / 'phase2_5'))
!python test_phase2_5_test2.py
os.chdir(str(WORK_DIR))
log.cell_end("Cell 12 — Test 2", "PASS")

---
## Cell 13: Test 3 — Unseen Logic Routing

In [ ]:
log.cell_start("Cell 13 — Test 3: Unseen Logic Routing")
os.chdir(str(WORK_DIR / 'phases' / 'phase2_5'))
!python test_phase2_5_test3.py
os.chdir(str(WORK_DIR))
log.cell_end("Cell 13 — Test 3", "PASS")

---
## Cell 14: Demo 1 — Sparse Galaxy (VRAM Efficiency)

In [ ]:
log.cell_start("Cell 14 — Demo 1: Sparse Galaxy")
os.chdir(str(WORK_DIR / 'phases' / 'phase2_5'))
!python demo_phase2_5_demo1.py

from IPython.display import Image, display
img = WORK_DIR / 'phases' / 'phase2_5' / 'demo2_5_sparse_galaxy.png'
if img.exists():
    display(Image(filename=str(img), width=950))
    log.success("Sparse galaxy visualization generated")

os.chdir(str(WORK_DIR))
log.cell_end("Cell 14 — Demo 1", "PASS")

---
## Cell 15: Demo 2 — Analogical Leaps

In [ ]:
log.cell_start("Cell 15 — Demo 2: Analogical Leaps")
os.chdir(str(WORK_DIR / 'phases' / 'phase2_5'))
!python demo_phase2_5_demo2.py

from IPython.display import Image, display
img2 = WORK_DIR / 'phases' / 'phase2_5' / 'demo2_5_analogical_leaps.png'
if img2.exists():
    display(Image(filename=str(img2), width=950))
    log.success("Analogical leaps visualization generated")

os.chdir(str(WORK_DIR))
log.cell_end("Cell 15 — Demo 2", "PASS")

---
## Cell 16: Results Summary

In [ ]:
log.cell_start("Cell 16 — Results Summary")

results_path = WORK_DIR / 'phases' / 'phase2_5' / 'RESULTS_PHASE_2_5.md'
if results_path.exists():
    from IPython.display import Markdown, display
    display(Markdown(results_path.read_text()))
    log.success("Results displayed")
else:
    # Generate inline summary
    status = field.active_capacity()
    print("\n" + "="*60)
    print("Phase 2.5 Summary")
    print("="*60)
    print(f"  Field tier:           {status['current_tier']} ({status['tier_label']})")
    print(f"  Dimensions:           {status['dimensions']}")
    print(f"  Active locations:     {status['active_locations']:,}")
    print(f"  Capacity used:        {status['capacity_fraction']:.4%}")
    print(f"  Sparse memory:        {status['sparse_memory_mb']:.1f} MB")
    print(f"  Growth events:        {status['growth_events']}")
    print(f"  Concepts seeded:      {seed_stats['total_seeded']}")
    print(f"  SNLI examples:        E={snli_stats['entailment']} C={snli_stats['contradiction']} N={snli_stats['neutral']}")
    print(f"  GSM8K problems:       {gsm8k_stats['problems_ingested']}")
    print(f"  Impl arrows total:    {len(impl_store.arrows)}")
    print("="*60)

# Print growth history
if field.growth_manager.growth_events:
    print("\n  Growth History:")
    for evt in field.growth_manager.growth_events:
        print(evt.log_str())
else:
    print("\n  No growth events (field capacity not exceeded at current seeding level)")

log.cell_end("Cell 16 — Results Summary", "PASS")

---
## Cell 17: View Full Log

In [ ]:
print("\n" + "="*60)
print("FULL PHASE 2.5 LOG")
print("="*60)
print(log.get_log_contents())

---
## Cell 18: Final Upload — Logs to HuggingFace & GitHub

In [ ]:
log.cell_start("Cell 18 — Final Upload")

upload_logs_to_hf(phase='2_5', hf_token=token)
log.success("Logs uploaded to HuggingFace Hub")

git_commit_and_push(
    files=[
        'logs/phase2_5.log',
        'phases/phase2_5/RESULTS_PHASE_2_5.md',
        'phases/phase2_5/demo2_5_sparse_galaxy.png',
        'phases/phase2_5/demo2_5_analogical_leaps.png',
    ],
    message='Phase 2.5: Ontological Bootstrapping complete [auto-commit from Kaggle]',
    repo_dir=str(WORK_DIR),
)

log.cell_end("Cell 18 — Final Upload", "PASS")

print("\n" + "="*60)
print("✓ PHASE 2.5 COMPLETE")
print("="*60)
print(f"  Checkpoint: https://huggingface.co/UnseenGAP/FLUX")
print(f"  Code:       https://github.com/Unseengap/FLUX")
print(f"  Next:       Phase 3 — Gravitational Relevance")
print("="*60)

---
## Cell 19: Save Artifacts to Kaggle Output

In [ ]:
import shutil

output_dir = Path('/kaggle/working/output')
output_dir.mkdir(exist_ok=True)

files_to_copy = [
    WORK_DIR / 'checkpoints' / 'phase2_5.phase.pt',
    WORK_DIR / 'phases' / 'phase2_5' / 'RESULTS_PHASE_2_5.md',
    WORK_DIR / 'logs' / 'phase2_5.log',
    WORK_DIR / 'phases' / 'phase2_5' / 'demo2_5_sparse_galaxy.png',
    WORK_DIR / 'phases' / 'phase2_5' / 'demo2_5_analogical_leaps.png',
]

# Also copy any tier checkpoints
for ckpt in (WORK_DIR / 'checkpoints').glob('phase2_5_tier*.pt'):
    files_to_copy.append(ckpt)

for f in files_to_copy:
    if Path(f).exists():
        shutil.copy(str(f), str(output_dir / Path(f).name))
        size = Path(f).stat().st_size / 1e6
        print(f'  ✓ {Path(f).name} → output/ ({size:.1f} MB)')
    else:
        print(f'  ⚠ {Path(f).name} not found')

print(f'\n  Output artifacts:')
for f in sorted(output_dir.iterdir()):
    print(f'    {f.name} ({f.stat().st_size / 1e6:.1f} MB)')

---
## Phase 2.5 Acceptance Criteria

| Criteria | Test | Target |
|---|---|---|
| Phase 2 coordinates migrate correctly | Test 1 | 100% feature match |
| Serialization round-trip | Test 1 | bit-exact |
| Contradiction pairs repelled | Test 2 | distance increase > 0% |
| Negative mass at contradiction locs | Test 2 | avg mass < 0.5 |
| Fictional word routing | Test 3 | ≥ 3/5 produce match |
| Analogy pairs | Test 3 | ≥ 2/7 correct |
| Sparse memory << dense | Demo 1 | < 1 GB vs 34 GB |
| Dynamic growth checkpoints | Cell 7-9 | saved at each tier |
| All checkpoints on HuggingFace | Cell 10 | ✓ |

**When all criteria pass → Phase 2.5 complete → Phase 3 unlocked!**